In [ ]:
# -*- coding: utf-8 -*-
"""
Korea POI Crawler (CITIES lat/lon) - Category-based, Single CSV, Keep importance_score + osm_url
- Overpass로 도시별 × 카테고리별 POI 수집(각 카테고리 30개)
- ✅ 결과는 "한 파일(통합 CSV)"로만 저장
- ✅ importance_score 컬럼 포함
- ✅ osm_url 컬럼 포함

최종 CSV 컬럼:
country, city, feature, name, lat, lon, importance_score, osm_url

필요: pip install requests pandas
"""

import os
import time
import math
import re
import requests
import pandas as pd


# ============================================================
# 1) 설정
# ============================================================
CITIES = [
    {"city": "베네치아", "lat": 45.4408474,  "lon": 12.3155151},
    {"city": "피렌체",   "lat": 43.652247, "lon": 10.945232},
    {"city": "로마", "lat": 41.9027835, "lon": 12.4963655}
]

COUNTRY = "대한민국"
TARGET_PER_CITY_PER_FEATURE = 30

RADIUS_STEPS = [5000, 10000, 20000, 40000, 80000]
ANCHOR_RADIUS_STEPS = [8000, 16000, 32000, 64000, 80000]

SLEEP_SEC = 0.25
MIN_DIST_BETWEEN_PICKED_M = 300

OUT_CSV = r"C:\ai\travel_dataset\나라도시\이탈리아.csv"
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

RUN_FEATURES = None

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

SESSION = requests.Session()


# ============================================================
# 2) 필터(숙박/노이즈 최소)
# ============================================================
LODGE_TOURISM = {
    "hotel", "hostel", "guest_house", "motel", "apartment", "resort",
    "chalet", "camp_site", "caravan_site"
}

NAME_BAD_RE = re.compile(
    r"(hotel|khách\s*sạn|resort|hostel|motel|homestay|apartment|villa|inn|"
    r"폐업|입구|놀이터)",
    re.I
)


# ============================================================
# 3) 카테고리(스포츠 없음)
# ============================================================
CATEGORIES = [
    {"feature": "먹거리", "key": "food",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="restaurant"]']},
         {"kind": "nwr", "tags": ['["amenity"="marketplace"]']},
         {"kind": "nwr", "tags": ['["amenity"="food_court"]']},
     ]},

    {"feature": "카페/베이커리", "key": "cafe_bakery",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="cafe"]']},
         {"kind": "nwr", "tags": ['["shop"="bakery"]']},
     ]},

    {"feature": "펍/바", "key": "pub_bar",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="pub"]']},
         {"kind": "nwr", "tags": ['["amenity"="bar"]']},
     ]},

    {"feature": "액티비티", "key": "activity",
     "blocks": [
         # 아웃도어
         {"kind": "relation", "tags": ['["route"="hiking"]']},
         {"kind": "way", "tags": ['["highway"~"^(path|footway|track)$"]']},

         {"kind": "nwr", "tags": ['["sport"="climbing"]']},
         {"kind": "nwr", "tags": ['["climbing"]']},

         {"kind": "nwr", "tags": ['["sport"~"^(surfing|kitesurfing|windsurfing|sailing|rowing|canoe|kayaking|diving|scuba_diving|swimming)$"]']},
         {"kind": "nwr", "tags": ['["leisure"="marina"]']},

         # 레저시설
         {"kind": "nwr", "tags": ['["tourism"="theme_park"]']},
         {"kind": "nwr", "tags": ['["tourism"="zoo"]']},
         {"kind": "nwr", "tags": ['["tourism"="aquarium"]']},
         {"kind": "nwr", "tags": ['["leisure"="bowling_alley"]']},
         {"kind": "nwr", "tags": ['["leisure"="amusement_arcade"]']},
     ]},

    {"feature": "쇼핑", "key": "shopping",
     "blocks": [
         {"kind": "nwr", "tags": ['["shop"="mall"]']},
         {"kind": "nwr", "tags": ['["shop"="department_store"]']},
         {"kind": "nwr", "tags": ['["shop"~"^(gift|craft|handicraft)$"]']},
     ]},

    {"feature": "자연/관광", "key": "nature_sightseeing",
     "blocks": [
         # 랜드마크/역사
         {"kind": "nwr", "tags": ['["tourism"="attraction"]']},
         {"kind": "nwr", "tags": ['["historic"="monument"]']},
         {"kind": "nwr", "tags": ['["historic"~"^(castle|palace|ruins|memorial|fort|archaeological_site)$"]']},

         # 자연 포인트
         {"kind": "nwr", "tags": ['["natural"="water"]["water"="lake"]']},
         {"kind": "nwr", "tags": ['["waterway"="waterfall"]']},
         {"kind": "nwr", "tags": ['["natural"="waterfall"]']},
         {"kind": "nwr", "tags": ['["natural"="peak"]']},
         {"kind": "nwr", "tags": ['["waterway"~"^(river|riverbank)$"]']},

         # 공원(식물원/정원/국립공원)
         {"kind": "nwr", "tags": ['["tourism"="botanical_garden"]']},
         {"kind": "nwr", "tags": ['["leisure"="garden"]']},
         {"kind": "relation", "tags": ['["boundary"="national_park"]']},

         # 웰니스
         {"kind": "nwr", "tags": ['["amenity"="spa"]']},
         {"kind": "nwr", "tags": ['["leisure"="sauna"]']},
         {"kind": "nwr", "tags": ['["natural"="hot_spring"]']},
     ]},
]

KEEP_COLS = ["country", "city", "feature", "name", "lat", "lon", "importance_score", "osm_url"]


# ============================================================
# 4) 유틸
# ============================================================
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def normalize_name(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s가-힣]", "", s)
    return s

def importance_score(tags: dict) -> int:
    s = 0
    if "wikidata" in tags: s += 50
    if "wikipedia" in tags: s += 40
    if "heritage" in tags: s += 25
    if "website" in tags: s += 5
    return int(s)

def pick_name(tags: dict) -> str:
    return (tags.get("name") or tags.get("name:ko") or tags.get("name:en") or "").strip()


# ============================================================
# 5) Overpass
# ============================================================
def build_overpass_query(lat: float, lon: float, radius_m: int, blocks: list):
    lines = ["[out:json][timeout:60];", "("]
    for b in blocks:
        kind = b.get("kind", "nwr")
        tag_str = "".join(b.get("tags", []))

        need_name = (kind in ["way", "relation"]) or ('["highway"' in tag_str) or ('["route"' in tag_str)
        name_filter = '["name"]' if need_name else ""

        lines.append(f'  {kind}(around:{radius_m},{lat},{lon}){tag_str}{name_filter};')
    lines.append(");")
    lines.append("out center tags;")
    return "\n".join(lines)

def overpass_fetch(query: str, max_retry=3):
    last_err = None
    for endpoint in OVERPASS_URLS:
        for attempt in range(1, max_retry + 1):
            try:
                r = SESSION.post(endpoint, data={"data": query}, timeout=120)
                if r.status_code == 200:
                    return r.json()
                last_err = f"{endpoint} status={r.status_code} body={r.text[:200]}"
                time.sleep(1.0 * attempt)
            except Exception as e:
                last_err = f"{endpoint} error={e}"
                time.sleep(1.0 * attempt)
    raise RuntimeError(f"❌ Overpass 요청 실패: {last_err}")

def parse_elements(data):
    elements = data.get("elements", []) if isinstance(data, dict) else []
    out = []
    for e in elements:
        tags = e.get("tags", {}) or {}

        name = pick_name(tags)
        if not name:
            continue
        if NAME_BAD_RE.search(name):
            continue

        tourism = (tags.get("tourism") or "").lower()
        if tourism in LODGE_TOURISM:
            continue

        lat = e.get("lat"); lon = e.get("lon")
        if lat is None or lon is None:
            center = e.get("center") or {}
            lat, lon = center.get("lat"), center.get("lon")
        if lat is None or lon is None:
            continue

        osm_type = e.get("type")
        osm_id = e.get("id")

        out.append({
            "name": name,
            "lat": float(lat),
            "lon": float(lon),
            "tags": dict(tags),
            "osm_type": osm_type,
            "osm_id": osm_id,
            "osm_url": f"https://www.openstreetmap.org/{osm_type}/{osm_id}",
        })
    return out


# ============================================================
# 6) 선별/중복 제거
# ============================================================
def too_close_to_picked(picked_rows, lat, lon, min_dist_m):
    for p in picked_rows:
        if haversine_m(p["lat"], p["lon"], lat, lon) < min_dist_m:
            return True
    return False

def rank_and_pick(city_name: str, clat: float, clon: float, feature: str, candidates: list, target_n: int):
    for c in candidates:
        c["_distance_m"] = haversine_m(clat, clon, c["lat"], c["lon"])
        c["_importance"] = importance_score(c["tags"])
    candidates.sort(key=lambda x: (-x["_importance"], x["_distance_m"]))

    picked = []
    seen_osm = set()
    seen_name_loc = set()

    for c in candidates:
        if len(picked) >= target_n:
            break

        uniq_osm = (c["osm_type"], c["osm_id"])
        if uniq_osm in seen_osm:
            continue

        key2 = (normalize_name(c["name"]), round(c["lat"], 5), round(c["lon"], 5))
        if key2 in seen_name_loc:
            continue

        if too_close_to_picked(picked, c["lat"], c["lon"], MIN_DIST_BETWEEN_PICKED_M):
            continue

        seen_osm.add(uniq_osm)
        seen_name_loc.add(key2)

        picked.append({
            "country": COUNTRY,
            "city": city_name,
            "feature": feature,
            "name": c["name"],
            "lat": c["lat"],
            "lon": c["lon"],
            "importance_score": int(c["_importance"]),
            "osm_url": c["osm_url"],
        })

    return picked


# ============================================================
# 7) 수집(도시×카테고리)
# ============================================================
def collect_city_feature(city_obj: dict, cat: dict):
    city = city_obj["city"]
    clat = float(city_obj["lat"])
    clon = float(city_obj["lon"])

    feature = cat["feature"]
    blocks = cat["blocks"]

    anchors = city_obj.get("anchors")
    centers = []
    if anchors:
        for a in anchors:
            centers.append((a["city"], float(a["lat"]), float(a["lon"]), ANCHOR_RADIUS_STEPS))
    else:
        centers.append((city, clat, clon, RADIUS_STEPS))

    all_candidates = []
    uniq = set()

    for center_name, lat, lon, steps in centers:
        for radius in steps:
            q = build_overpass_query(lat, lon, radius, blocks)
            data = overpass_fetch(q, max_retry=2)
            cand = parse_elements(data)

            added = 0
            for c in cand:
                k = (c["osm_type"], c["osm_id"])
                if k in uniq:
                    continue
                uniq.add(k)
                all_candidates.append(c)
                added += 1

            print(f"  - {city}/{feature} anchor={center_name} radius={radius}m +{added} (누적 {len(all_candidates)})")
            time.sleep(SLEEP_SEC)

            if len(all_candidates) >= max(80, TARGET_PER_CITY_PER_FEATURE * 3):
                break

    return rank_and_pick(city, clat, clon, feature, all_candidates, TARGET_PER_CITY_PER_FEATURE)


# ============================================================
# 8) 실행 + 통합 CSV 저장
# ============================================================
def main():
    all_rows = []

    for city_obj in CITIES:
        print("\n" + "=" * 70)
        print(f"▶ 도시 시작: {city_obj['city']}")

        for cat in CATEGORIES:
            feature = cat["feature"]
            if RUN_FEATURES is not None and feature not in RUN_FEATURES:
                continue

            print(f"\n▶ {city_obj['city']} / {feature} 수집 시작 (목표 {TARGET_PER_CITY_PER_FEATURE})")
            try:
                rows = collect_city_feature(city_obj, cat)
                print(f"✅ {city_obj['city']} / {feature} 완료: {len(rows)}건")
                all_rows.extend(rows)

                # ✅ 중간 저장
                df_tmp = pd.DataFrame(all_rows)
                df_tmp = df_tmp[KEEP_COLS]
                df_tmp.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
                print(f"📁 중간 저장: {OUT_CSV} (누적 {len(df_tmp)}건)")
            except Exception as e:
                print(f"❌ {city_obj['city']} / {feature} 실패(스킵): {e}")
                continue

    if not all_rows:
        print("\n❌ 최종 0건입니다. (Overpass 장애/네트워크/차단 가능)")
        return

    df = pd.DataFrame(all_rows)
    df = df[KEEP_COLS].copy()
    df = df.sort_values(["city", "feature", "importance_score"], ascending=[True, True, False])

    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ 최종 CSV 저장 완료: {OUT_CSV}")
    print(df.groupby(["city", "feature"])["name"].count())

if __name__ == "__main__":
    main()
